<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 4 · Software Engineering, Reproducibility, and Deployment Basics
### Notebook 03 · Environments, Dashboard Apps, and Deployment Awareness
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
A project is easier to share when someone else can install it without guessing which
packages or versions are needed.

This cell makes the notebook portable. It finds the book root locally, or clones
`tdscode` in Colab, so the rest of the notebook can use stable paths.

This cell checks the Python version and dependency list so the learner can compare
the runtime with the book requirements.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "4_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


In [ ]:
import platform
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
BOOK_ROOT = cwd
for candidate in (cwd, cwd.parent, cwd / "4_data"):
    if (candidate / "src").exists() or (candidate / "code").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

req_path = BOOK_ROOT / "requirements.txt"
requirements = [
    line.strip()
    for line in req_path.read_text().splitlines()
    if line.strip() and not line.startswith("#")
]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("requirements.txt:")
print("\n".join(f"- {item}" for item in requirements))


This cell prints the recommended install commands for local machines and Colab so the
setup path is explicit.

In [ ]:
print("Local install: python -m pip install -r requirements.txt")
print("Colab install: %pip install -q -r requirements.txt")
print(
    "Recommended local flow: create a virtual environment, activate it, "
    "and then install."
)


This cell connects the reusable project-health code to a small dashboard example and
a deployment recipe.

In [ ]:
from pathlib import Path
import os

from src.data_checks import (
    load_project_health,
    project_health_summary,
)

frame = load_project_health()
filtered = frame[
    (frame["domain"] == "finance") & (frame["owner_type"] == "team")
]
summary = project_health_summary(filtered)

print("Filtered rows:", len(filtered))
print(summary)
print()
columns = ["project_id", "stale_days", "issue_count", "delivery_risk"]
print(filtered[columns].to_string(index=False))

dashboard_source = BOOK_ROOT / "src" / "dashboard_app.py"
lines = dashboard_source.read_text().splitlines()
print()
print("Dashboard entry point snippet:")
print("\n".join(lines[:24]))

# Minimal deployment recipe.
dockerfile = "\n".join(
    [
        "FROM python:3.12-slim",
        "WORKDIR /app",
        "COPY requirements.txt .",
        "RUN pip install --no-cache-dir -r requirements.txt",
        "COPY . .",
        (
            'CMD ["streamlit", "run", "src/dashboard_app.py", '
            '"--server.port", "8501", '
            '"--server.address", "0.0.0.0"]'
        ),
    ]
)
print()
print(dockerfile)
print()
print("Local run command: streamlit run src/dashboard_app.py")
print("Do not hard-code secrets; read them from environment variables instead.")
print("Example API key present:", bool(os.environ.get("EXAMPLE_API_KEY")))


### Where We Are Heading Next
The capstone notebook ties the whole module together as one small, portfolio-ready
project.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">